## Install all the required packages

In [1]:
!pip install langchain
!pip install pinecone
!pip install pypdf



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.5/334.5 kB 5.6 MB/s eta 0:00:00


In [2]:
!pip install langchain_community
!pip install tiktoken
!pip install azure.ai.inference

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.9/124.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 17.4 MB/s eta 0:00:00


## Import All the required libraries

In [3]:
!pip -q install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 2.9 MB/s eta 0:00:00


In [4]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from typing import List
from azure.ai.inference import EmbeddingsClient
from azure.core.credentials import AzureKeyCredential
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import Pinecone
from langchain_classic.chains import RetrievalQA
from langchain_classic.prompts import PromptTemplate


Load the PDF Files

In [5]:
!mkdir pdfs
## upload your pdfs to the pdfs directory

## Extract the Text from the PDFs

In [6]:
loader = PyPDFDirectoryLoader("pdfs")
data = loader.load()

In [7]:
data

[Document(metadata={'producer': 'OpenOffice.org 3.0', 'creator': 'Writer', 'creationdate': '2009-02-20T16:34:34-08:00', 'title': 'In Him', 'author': 'Kenneth E. Hagin', 'keywords': "Jesus, Christ, Bible, spiritual warfare, believer's authority", 'subject': 'In Him', 'moddate': '2009-02-20T16:39:39-08:00', 'source': 'pdfs/In+Him.pdf', 'total_pages': 34, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'OpenOffice.org 3.0', 'creator': 'Writer', 'creationdate': '2009-02-20T16:34:34-08:00', 'title': 'In Him', 'author': 'Kenneth E. Hagin', 'keywords': "Jesus, Christ, Bible, spiritual warfare, believer's authority", 'subject': 'In Him', 'moddate': '2009-02-20T16:39:39-08:00', 'source': 'pdfs/In+Him.pdf', 'total_pages': 34, 'page': 1, 'page_label': '2'}, page_content="Copyright © 1980 RHEMA Bible Church\nAKA Kenneth Hagin Ministries, Inc.\nAll Rights Reserved\nPrinted in USA\nForty-First Printing 2004\nISBN 0-89276-052-4\nTo receive a free, full-color brochure 

## Split the Extracted Data Text chunks

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)

In [9]:
text_chunks = text_splitter.split_documents(data)

In [10]:
text_chunks

[Document(metadata={'producer': 'OpenOffice.org 3.0', 'creator': 'Writer', 'creationdate': '2009-02-20T16:34:34-08:00', 'title': 'In Him', 'author': 'Kenneth E. Hagin', 'keywords': "Jesus, Christ, Bible, spiritual warfare, believer's authority", 'subject': 'In Him', 'moddate': '2009-02-20T16:39:39-08:00', 'source': 'pdfs/In+Him.pdf', 'total_pages': 34, 'page': 1, 'page_label': '2'}, page_content="Copyright © 1980 RHEMA Bible Church\nAKA Kenneth Hagin Ministries, Inc.\nAll Rights Reserved\nPrinted in USA\nForty-First Printing 2004\nISBN 0-89276-052-4\nTo receive a free, full-color brochure on RHEMA Bible  \nTraining  Center;  a  free  monthly  magazine,  The  Word  of \nFaith; or to receive our Faith Library Catalog with a complete \nlisting of Kenneth Hagin Ministries' books and tapes, in the  \nU.S., write:\nKenneth Hagin Ministries\nP. O. Box 50126, Tulsa, OK 74150-0126\n1-888-28-FAITH www.rhema.org\nIn Canada write: P. O. Box 335, Station D\nEtobicoke, Ontario, Canada, M9A 4X3\nThe 

In [11]:
len(text_chunks)

218

## Download the Embeddings

In [12]:
import os

os.environ['GITHUB_TOKEN'] = "github_pat_"

In [13]:
import os
from typing import List
from azure.ai.inference import EmbeddingsClient
from azure.core.credentials import AzureKeyCredential

class AzureEmbeddingsWrapper:
    def __init__(self):
        self.endpoint = "https://models.github.ai/inference"
        self.model_name = "openai/text-embedding-3-large"
        self.token = os.environ["GITHUB_TOKEN"]

        self.client = EmbeddingsClient(
            endpoint=self.endpoint,
            credential=AzureKeyCredential(self.token)
        )

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        response = self.client.embed(
            input=texts,
            model=self.model_name
        )
        return [item.embedding for item in response.data]

    def embed_query(self, text: str) -> List[float]:
        response = self.client.embed(
            input=[text],
            model=self.model_name
        )
        return response.data[0].embedding

In [14]:
embedding = AzureEmbeddingsWrapper()

In [15]:
result = embedding.embed_query("Hello")

In [16]:
len(result)

3072

## Initializing the pinecone

In [17]:
!pip -q install langchain-pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.


In [18]:
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY", "")
PINECONE_API_ENV = os.environ.get("PINECONE_API_ENV", "gcp-starter")

In [19]:
from pinecone import Pinecone as PineconeClient
from langchain_pinecone import Pinecone

# init pinecone
pc = PineconeClient(api_key=PINECONE_API_KEY)

index_name = "test"

# connect to existing index
index = pc.Index(index_name)


# vector store
vectorstore = Pinecone(
    index=index,
    embedding=embedding
)

/tmp/ipykernel_4140/4109327232.py:14: LangChainDeprecationWarning: The class `Pinecone` was deprecated in LangChain 0.0.3 and will be removed in 1.0.0. Use `PineconeVectorStore` instead.
  vectorstore = Pinecone(


## Create embeddings for each of the chunks

In [20]:
# assuming text_chunks is a list of Documents, the code below is for first time indexing and adding new data
def batch_texts(texts, batch_size=30):  # 🔥 smaller batch
    for i in range(0, len(texts), batch_size):
        yield texts[i:i + batch_size]

texts = [t.page_content for t in text_chunks]

for batch in batch_texts(texts, 30):
    vectorstore.add_texts(batch)

Similarity Search

In [23]:
## create a retrieval
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [26]:
## query directly

docs = retriever.invoke("What is a believer authority?")

for doc in docs:
    print(doc.page_content)

The
Believer's
Authority
Kenneth E. Hagin
38 The Believer's Authority
Remember, Colossians 1:13 says,  "Who hath delivered us 
from the power of darkness, and hath translated  us into the 
kingdom of his dear Son...." (One translation says  "the Father 
hath delivered us from the power of darkness.") Again that's the 
Greek word 'power' here for 'authority.'
The verse should read, "The Father hath delivered us from 
the  authority  of  darkness,  and  hath  translated  us  into  the 
kingdom of his dear Son." God already has delivered us from the 
authority of darkness! Therefore, we've got a right to speak to 
darkness—that is, Satan and his kingdom—and tell them what to 
do!
Exercising Authority Over Others
Believers have authority over the devil. They can break the 
power of the devil if he raises his head anywhere in their own 
life or the lives of their immediate family or loved ones. They 
have authority there. They'll be free from the enemy because 
they've got the right to exer

## Create a LLM model Wrapper

In [27]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    base_url="https://models.github.ai/inference",
    api_key=os.environ["GITHUB_TOKEN"],
    model="gpt-4o-mini",  # or any available model
    temperature=0
)

In [28]:
llm

ChatOpenAI(profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x7a977842f9b0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7a97782c6ed0>, root_client=<openai.OpenAI object at 0x7a977af85e50>, root_async_client=<openai.AsyncOpenAI object at 0x7a977842f530>, model_name='gpt-4o-mini', temperature=0.0, model_kwargs={}, openai_api_key=Secre

In [29]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

In [32]:
query = "list 5 confessions we have in HIM"
llm_response = qa_chain({"query": query})
llm_response

{'query': 'list 5 confessions we have in HIM',
 'result': '1. "Because I am in Christ Jesus, RIGHT NOW, there is no sense of condemnation about me."\n2. "Christ Jesus, my Lord, is my wisdom. He is my righteousness. He is my sanctification. He is my redemption."\n3. "I have received abundance of grace and the gift of righteousness. I reign as a king in my domain in this life through Jesus Christ."\n4. "In Him I live, and move, and have my being!"\n5. "I abide in Him. I live in Him. His Life—the Life of God—is in me."',
 'source_documents': [Document(id='74f22bd0-0d73-4ee9-ada3-098e867900c1', metadata={}, page_content='5\nhe saith ..." (Mk. 11:23).\nHEBREWS 4:14\n14 Seeing then that we have a great high priest,  \nthat is passed into the heavens, Jesus the Son of  \nGod, let us hold fast our profession (confession).\nThe  same  Greek  word  translated  profession  here  is  \ntranslated elsewhere in the King James as confession. Modern \ntranslations  render  it  confession.  "Let  us  h